In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abdullahalsunny/eye-state/data.npz
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_225.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_419.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_269.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_86.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_707.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_163.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_19.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_194.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_652.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_580.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_330.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_212.jpg
/kaggle/input/datasets/abdullahalsunny/eye-state/Dataset/Closed/_500

In [2]:
!pip install -q timm albumentations scikit-learn tabulate

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# If the folder name is different, change it here
data_path = '/kaggle/input/eyestate-recognition-dataset/data.npz'
data = np.load(data_path)

print("Keys in .npz file:", data.files)

if 'images' in data.files:
    images = data['images']
    labels = data['labels']
elif 'X' in data.files:
    images = data['X']
    labels = data['y']
elif 'arr_0' in data.files:
    images = data['arr_0']
    labels = data['arr_1']
else:
    raise KeyError(f"Unknown keys: {data.files}")

print(f"Images shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {np.unique(labels)} (0=Closed, 1=Open)")

labels = labels.astype(np.int64)

# Convert grayscale to 3-channel
if images.ndim == 3:
    images = np.stack([images] * 3, axis=-1)
elif images.ndim == 4 and images.shape[-1] == 1:
    images = np.squeeze(images, axis=-1)
    images = np.stack([images] * 3, axis=-1)

if images.max() > 1.0:
    images = images / 255.0

print(f"Final images shape: {images.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/eyestate-recognition-dataset/data.npz'

In [ ]:

# ============================================
# Cell 2: Auto-Find data.npz (No hardcoded path)
# ============================================
import os
import glob

# Search for data.npz anywhere inside /kaggle/input/
search_pattern = '/kaggle/input/**/data.npz'
found_files = glob.glob(search_pattern, recursive=True)

if not found_files:
    # Try alternative name (some datasets name it differently)
    found_files = glob.glob('/kaggle/input/**/*.npz', recursive=True)

if found_files:
    data_path = found_files[0]
    print(f"✅ Found file at: {data_path}")
else:
    # List all files in input so you can see what's there
    print("❌ No .npz file found. Here are all files in /kaggle/input/:")
    for root, dirs, files in os.walk('/kaggle/input/'):
        for file in files:
            print(f"   {os.path.join(root, file)}")
    raise FileNotFoundError("Could not find data.npz. Please check the file name and try again.")

# Load the file
data = np.load(data_path)

print("Keys in .npz file:", data.files)

if 'images' in data.files:
    images = data['images']
    labels = data['labels']
elif 'X' in data.files:
    images = data['X']
    labels = data['y']
elif 'arr_0' in data.files:
    images = data['arr_0']
    labels = data['arr_1']
else:
    raise KeyError(f"Unknown keys: {data.files}")

print(f"Images shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {np.unique(labels)} (0=Closed, 1=Open)")

labels = labels.astype(np.int64)

# Convert grayscale to 3-channel
if images.ndim == 3:
    images = np.stack([images] * 3, axis=-1)
elif images.ndim == 4 and images.shape[-1] == 1:
    images = np.squeeze(images, axis=-1)
    images = np.stack([images] * 3, axis=-1)

if images.max() > 1.0:
    images = images / 255.0

print(f"Final images shape: {images.shape}")



In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.30, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

X_train_t = torch.FloatTensor(X_train).permute(0, 3, 1, 2)
X_val_t   = torch.FloatTensor(X_val).permute(0, 3, 1, 2)
X_test_t  = torch.FloatTensor(X_test).permute(0, 3, 1, 2)
y_train_t = torch.LongTensor(y_train)
y_val_t   = torch.LongTensor(y_val)
y_test_t  = torch.LongTensor(y_test)

class EyeDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

class NormalizeOnly:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, tensor):
        return transforms.Normalize(mean=self.mean, std=self.std)(tensor)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    NormalizeOnly(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = NormalizeOnly(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_dataset = EyeDataset(X_train_t, y_train_t, transform=train_transform)
val_dataset   = EyeDataset(X_val_t, y_val_t, transform=val_transform)
test_dataset  = EyeDataset(X_test_t, y_test_t, transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Cell 4 – No pretrained weights (offline mode)
import torchvision.models as tv_models

NUM_CLASSES = 2
EPOCHS = 15
LEARNING_RATE = 1e-3   # slightly higher LR for training from scratch

# Use torchvision's EfficientNet-B0 WITHOUT pretrained weights
model = tv_models.efficientnet_b0(pretrained=False)
# Replace classifier for 2 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print("Model: EfficientNet-B0 (trained from scratch)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ---- Training loop (same as before) ----
train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    
    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    scheduler.step(epoch_val_loss)
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"*** Best model saved (Val Acc: {best_val_acc:.4f}) ***")
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_train_loss:.4f}, Train Acc={epoch_train_acc:.4f} | Val Loss={epoch_val_loss:.4f}, Val Acc={epoch_val_acc:.4f}")

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='binary')
recall = recall_score(all_labels, all_preds, average='binary')
f1 = f1_score(all_labels, all_preds, average='binary')
auc = roc_auc_score(all_labels, all_probs)

print("="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")
print("="*50)

# ---- Charts ----
plt.style.use('seaborn-v0_8-darkgrid')
fig = plt.figure(figsize=(16, 12))

plt.subplot(2, 2, 1)
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Closed', 'Open'], 
            yticklabels=['Closed', 'Open'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('True')

plt.subplot(2, 2, 2)
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}', linewidth=2, color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')

plt.subplot(2, 2, 3)
plt.plot(range(1, EPOCHS+1), train_losses, marker='o', label='Train Loss', color='blue')
plt.plot(range(1, EPOCHS+1), val_losses, marker='s', label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
plt.plot(range(1, EPOCHS+1), train_accs, marker='o', label='Train Accuracy', color='blue')
plt.plot(range(1, EPOCHS+1), val_accs, marker='s', label='Val Accuracy', color='red')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curves', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/all_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("Charts saved to /kaggle/working/all_charts.png")

In [ ]:
# ============================================
# Zip all results for easy download
# ============================================
import os
import zipfile

# Files to include
files_to_zip = [
    '/kaggle/working/best_model.pth',
    '/kaggle/working/all_charts.png'
]

# Create a zip file
zip_path = '/kaggle/working/results.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file, os.path.basename(file))
            print(f"✅ Added: {os.path.basename(file)}")
        else:
            print(f"⚠️ File not found: {file}")

# Also add the notebook output as a text file (optional)
with open('/kaggle/working/metrics_summary.txt', 'w') as f:
    f.write("=== FairEye Results ===\n")
    f.write(f"Validation Accuracy: {best_val_acc:.4f}\n")
    # You can add test metrics here manually after Cell 5 prints them

zipf.close()
print(f"\n✅ All results zipped to: {zip_path}")
print("📁 File size:", os.path.getsize(zip_path) / 1024, "KB")

# List all files in /kaggle/working/
print("\n=== Files in /kaggle/working/ ===")
for file in os.listdir('/kaggle/working/'):
    size = os.path.getsize(f'/kaggle/working/{file}')
    print(f"  {file} ({size/1024:.1f} KB)")

In [ ]:
# ============================================
# Pie Charts (Fixed)
# ============================================
import matplotlib.pyplot as plt
import numpy as np

# --- Convert labels to NumPy if needed ---
# labels is from Cell 2 – convert to numpy
if hasattr(labels, 'numpy'):
    labels_np = labels.numpy()
else:
    labels_np = np.array(labels)

# --- Pie Chart 1: Dataset Class Distribution ---
total_open = np.sum(labels_np == 1)
total_closed = np.sum(labels_np == 0)
total_images = len(labels_np)

print(f"Dataset Distribution:")
print(f"  Closed: {total_closed} ({total_closed/total_images*100:.1f}%)")
print(f"  Open:   {total_open} ({total_open/total_images*100:.1f}%)")

# --- Pie Chart 2: Test Set Predictions (Correct vs Incorrect) ---
# Using all_preds and all_labels from Cell 5
if 'all_preds' in locals() and 'all_labels' in locals():
    correct = np.sum(np.array(all_preds) == np.array(all_labels))
    incorrect = len(all_labels) - correct
    total_test = len(all_labels)
    print(f"\nTest Set Predictions:")
    print(f"  Correct:   {correct} ({correct/total_test*100:.1f}%)")
    print(f"  Incorrect: {incorrect} ({incorrect/total_test*100:.1f}%)")
else:
    # Fallback: use validation accuracy as estimate
    correct = int(best_val_acc * 100)
    incorrect = 100 - correct
    total_test = 100
    print("\nTest Set Predictions (estimated from validation accuracy):")
    print(f"  Correct:   {correct}%")
    print(f"  Incorrect: {incorrect}%")

# --- Create the Figure ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie 1: Class Distribution
colors1 = ['#ff6b6b', '#4ecdc4']
explode1 = (0.05, 0.05)
axes[0].pie(
    [total_closed, total_open],
    labels=['Closed', 'Open'],
    autopct='%1.1f%%',
    colors=colors1,
    explode=explode1,
    startangle=90,
    shadow=True,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[0].set_title('Dataset Class Distribution', fontsize=14, fontweight='bold')

# Pie 2: Correct vs Incorrect Predictions
colors2 = ['#2ecc71', '#e74c3c']
explode2 = (0.05, 0.05)
axes[1].pie(
    [correct, incorrect],
    labels=['Correct', 'Incorrect'],
    autopct='%1.1f%%',
    colors=colors2,
    explode=explode2,
    startangle=90,
    shadow=True,
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[1].set_title('Test Set Predictions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/pie_charts.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Pie charts saved to: /kaggle/working/pie_charts.png")

In [ ]:
# ============================================
# Bar Chart: Model Performance Metrics
# ============================================
import matplotlib.pyplot as plt
import numpy as np

# Use your actual metrics from Cell 5
# Replace these with your exact numbers:
metrics = {
    'Accuracy': 0.9723,
    'Precision': 0.9741,
    'Recall': 0.9705,
    'F1-Score': 0.9723,
    'AUC-ROC': 0.9948
}

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='black', linewidth=1.2)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(0.85, 1.02)
plt.ylabel('Score', fontsize=14, fontweight='bold')
plt.title('FairEye Model Performance (EfficientNet-B0)', fontsize=16, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tick_params(axis='x', labelsize=12)
plt.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5, linewidth=1)

plt.tight_layout()
plt.savefig('/kaggle/working/performance_bar_chart.png', dpi=200, bbox_inches='tight')
plt.show()

print("✅ Bar chart saved to: /kaggle/working/performance_bar_chart.png")